### An Invitation to Analytic Combinatorics - Lecture 4

This SageMath notebook contains the Sage code presented in Lecture 4 of the MPI Lecture Series presented by Stephen Melczer in June 2026.

### Part 1: An Explicit Example

In [1]:
# We numerically approximate I - I_loc by reducing to a 1D integral with 
# integrand 1 / (1-x)^(n+1)/x^(n+1). In this example I_out = 0 so
# chi = I_loc. Change n here to get different approximations.

# Sage has some peculiarities with setting up the CBF
CBF = ComplexBallField(300)
I_ball = CBF(0, 1)
TWO_PI = 2 * CBF.pi()
HALF = CBF("0.5")

def I_loc_approx(n):
    def integrand_1D(theta, analytic=False):
        x = HALF * (I_ball * theta).exp()
        dx = I_ball * x
        return (1 / (1 - x)^(n+1)) / (x^(n+1)) * dx

    return (CBF.integral(integrand_1D, -CBF.pi()/4, CBF.pi()/4)/ (TWO_PI * I_ball)).real()

In [2]:
n = 10
print(binomial(2*n,n))
print(I_loc_approx(n))

184756
[185571.9969689691004905405251972240758864536144356542671409185828864473287853534103395391 +/- 5.59e-83]


In [3]:
n = 20
print(binomial(2*n,n))
print(I_loc_approx(n))

137846528820
[137837443284.7551466659921067026785688473331032950955403516705253793106859592671107613086 +/- 8.57e-77]


In [4]:
n = 100
print(binomial(2*n,n))
print(I_loc_approx(n))

90548514656103281165404177077484163874504589675413336841320
[90548514656103281244226514758732425815049173009882161476501.104725254454145676582503065 +/- 4.69e-28]


In [5]:
# We can numerically approximate the exponential growth of I - I_loc,
# which in this case equals I - chi
def approx_exp(n):
    return exp(log(I_loc_approx(n) - binomial(2*n,n))/n).n()

print(approx_exp(50))
print(approx_exp(100))

2.42856201685023
2.56428618787537


In [6]:
# If a_n ~ C(1 + A/n + ...) then 2a_{2n}-a_n converges quadratically to C
# So this (heuristically) gives a better approximation of the exponential growth
# Recall we had an error bound O(2.72^n) for I - I_loc, so already at n=50 we get
# close to our bound.
2*approx_exp(100) - approx_exp(50)

2.70001035890050

### Part 2: Theory of Smooth ACSV

In [7]:
# Example with extra critical points from bounded amoeba component
var('x,y')
H = 1 - x - y - 6*x*y - x^2*y^2

In [8]:
solve([H,x*H.derivative(x)-y*H.derivative(y)])

[[x == 0.2732372569881033, y == 0.2732372569881033],
 [x == -0.5849196082055073, y == -0.5849196082055073],
 [x == (0.15584117177618814 + 2.496533844704942*I),
  y == (0.15584117177618814 + 2.496533844704942*I)],
 [x == (0.15584117177618814 - 2.496533844704942*I),
  y == (0.15584117177618814 - 2.496533844704942*I)]]

In [9]:
# NSEW lattice path model asymptotics
var('x,y,t')
G = (1+x)*(1+y)
H = 1 - t*x*y*(x+1/x+y+1/y)
solve([H, t*H.derivative(t) - x*H.derivative(x), t*H.derivative(t) - y*H.derivative(y)])

[[x == 1, y == 1, t == (1/4)], [x == -1, y == -1, t == (-1/4)]]

In [10]:
g = solve(H,t)[0].rhs()
g

1/(x*y^2 + (x^2 + 1)*y + x)

In [11]:
var('T1,T2')
Hes = log(g).substitute(x=exp(I*T1), y=exp(I*T2)).hessian()
HesDet = Hes.determinant().substitute(T1=0,T2=0)
HesDet

1/4

In [12]:
var('n')
d = 3
ASM = (x*y*t)^(-n) * n^((1-d)/2) * (2*pi)^((1-d)/2)/sqrt(HesDet) * (-G)/t/H.derivative(t)
ASM.subs(x==1,y==1,t==1/4).simplify()

4^(n + 1)/(pi*n)

In [13]:
# Now do this automatically in sage_acsv
# This requires installing sage_acsv
# See https://github.com/ACSVMath/sage_acsv
from sage_acsv import diagonal_asymptotics_combinatorial as diagonal

In [14]:
# First term in asymptotic expansion
diagonal(G/H)

4/pi*4^n*n^(-1) + O(4^n*n^(-2))

In [15]:
# First 3 terms in expansion
diagonal(G/H, expansion_precision=3)

4/pi*4^n*n^(-1) - 6/pi*4^n*n^(-2) + 1/pi*4^n*n^(-3)*(e^(I*arg(-1)))^n + 19/2/pi*4^n*n^(-3) + O(4^n*n^(-4))